# 08 - Final Analysis

This notebook combines the results of the four experiments performed in
Practical Application 1:

1. All valid features.
2. Univariate filter feature selection.
3. Multivariate filter feature selection.
4. Wrapper feature selection.

The objective is to compare predictive performance, analyse the effect of
feature selection, identify consistently selected employee characteristics,
and generate the final tables and figures used in the scientific report.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 1. Load Experimental Results

The results from the four feature-selection experiments are combined to
compare the five classifiers. All experiments use the same dataset, outer
cross-validation folds, and evaluation metrics.

In [2]:
baseline = pd.read_csv(
    "../../results/pa1/baseline_summary.csv"
)

univariate = pd.read_csv(
    "../../results/pa1/univariate_summary.csv"
)

multivariate = pd.read_csv(
    "../../results/pa1/multivariate_summary.csv"
)

wrapper = pd.read_csv(
    "../../results/pa1/wrapper_summary.csv"
)

baseline["Experiment"] = "All Features"
univariate["Experiment"] = "Univariate"
multivariate["Experiment"] = "Multivariate"
wrapper["Experiment"] = "Wrapper"

all_results = pd.concat(
    [
        baseline,
        univariate,
        multivariate,
        wrapper
    ],
    ignore_index=True
)

display(all_results.round(4))


expected_models = {"K-NN", "Decision Tree", "SVM", "Neural Network", "RIPPER"}
for experiment, group in all_results.groupby("Experiment"):
    assert len(group) == 5 and set(group["Model"]) == expected_models, experiment


,Model,Accuracy,Precision,Recall,F1,Balanced Accuracy,Experiment,Selected Features
0,Neural Network,0.8490,0.5492,0.4219,0.4738,0.6765,All Features,NaN
1,Decision Tree,0.7925,0.3604,0.3670,0.3620,0.6206,All Features,NaN
2,SVM,0.8653,0.8363,0.2241,0.3461,0.6064,All Features,NaN
3,RIPPER,0.8306,0.4507,0.2147,0.2866,0.5818,All Features,NaN
4,K-NN,0.8442,0.5412,0.1350,0.2115,0.5577,All Features,NaN
5,Decision Tree,0.7789,0.3546,0.4219,0.3809,0.6347,Univariate,15.0
6,Neural Network,0.8163,0.4167,0.3293,0.3642,0.6197,Univariate,15.0
7,K-NN,0.8463,0.5761,0.2062,0.2968,0.5877,Univariate,15.0
8,SVM,0.8571,0.7933,0.1692,0.2737,0.5793,Univariate,15.0
9,RIPPER,0.8415,0.6102,0.1558,0.2316,0.5645,Univariate,15.0


## 2. Unified Results Table

The four experiments are organized using the same set of columns to facilitate
direct comparison between feature-selection strategies and classifiers.

In [3]:
# Feature counts come directly from the experiment summaries.
# Baseline representation differs by model; selected counts can be fold means.
all_results["Number of Features"] = all_results["Selected Features"].astype(float)
baseline_mask = all_results["Experiment"].eq("All Features")
all_results.loc[baseline_mask, "Number of Features"] = (
    all_results.loc[baseline_mask, "Model"].map(
        lambda model: 30 if model == "RIPPER" else 51
    )
)

all_results = all_results[
    [
        "Experiment",
        "Model",
        "Number of Features",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Balanced Accuracy"
    ]
]

all_results = all_results.sort_values(
    ["F1", "Balanced Accuracy"],
    ascending=False
)

display(
    all_results.round(4)
)

,Experiment,Model,Number of Features,Accuracy,Precision,Recall,F1,Balanced Accuracy
0,All Features,Neural Network,51.0,0.8490,0.5492,0.4219,0.4738,0.6765
15,Wrapper,RIPPER,2.6,0.7823,0.3640,0.4687,0.3996,0.6557
16,Wrapper,Neural Network,15.0,0.8401,0.5278,0.3165,0.3919,0.6286
10,Multivariate,Neural Network,15.0,0.8245,0.4501,0.3534,0.3902,0.6341
5,Univariate,Decision Tree,15.0,0.7789,0.3546,0.4219,0.3809,0.6347
11,Multivariate,Decision Tree,15.0,0.7898,0.3652,0.3958,0.3748,0.6306
6,Univariate,Neural Network,15.0,0.8163,0.4167,0.3293,0.3642,0.6197
1,All Features,Decision Tree,51.0,0.7925,0.3604,0.3670,0.3620,0.6206
17,Wrapper,Decision Tree,15.0,0.7782,0.3358,0.3879,0.3594,0.6205
2,All Features,SVM,51.0,0.8653,0.8363,0.2241,0.3461,0.6064


## 3. Best Overall Configurations

The configurations are ranked primarily by F1-score because the positive
attrition class is a minority class.

Balanced accuracy is also considered because it evaluates performance across
both target classes.

In [4]:
best_by_f1 = (
    all_results
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

display(
    best_by_f1.head(10).round(4)
)

,Experiment,Model,Number of Features,Accuracy,Precision,Recall,F1,Balanced Accuracy
0,All Features,Neural Network,51.0,0.8490,0.5492,0.4219,0.4738,0.6765
1,Wrapper,RIPPER,2.6,0.7823,0.3640,0.4687,0.3996,0.6557
2,Wrapper,Neural Network,15.0,0.8401,0.5278,0.3165,0.3919,0.6286
3,Multivariate,Neural Network,15.0,0.8245,0.4501,0.3534,0.3902,0.6341
4,Univariate,Decision Tree,15.0,0.7789,0.3546,0.4219,0.3809,0.6347
5,Multivariate,Decision Tree,15.0,0.7898,0.3652,0.3958,0.3748,0.6306
6,Univariate,Neural Network,15.0,0.8163,0.4167,0.3293,0.3642,0.6197
7,All Features,Decision Tree,51.0,0.7925,0.3604,0.3670,0.3620,0.6206
8,Wrapper,Decision Tree,15.0,0.7782,0.3358,0.3879,0.3594,0.6205
9,All Features,SVM,51.0,0.8653,0.8363,0.2241,0.3461,0.6064


In [5]:
best_by_balanced_accuracy = (
    all_results
    .sort_values(
        "Balanced Accuracy",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    best_by_balanced_accuracy.head(10).round(4)
)

,Experiment,Model,Number of Features,Accuracy,Precision,Recall,F1,Balanced Accuracy
0,All Features,Neural Network,51.0,0.8490,0.5492,0.4219,0.4738,0.6765
1,Wrapper,RIPPER,2.6,0.7823,0.3640,0.4687,0.3996,0.6557
2,Univariate,Decision Tree,15.0,0.7789,0.3546,0.4219,0.3809,0.6347
3,Multivariate,Neural Network,15.0,0.8245,0.4501,0.3534,0.3902,0.6341
4,Multivariate,Decision Tree,15.0,0.7898,0.3652,0.3958,0.3748,0.6306
5,Wrapper,Neural Network,15.0,0.8401,0.5278,0.3165,0.3919,0.6286
6,All Features,Decision Tree,51.0,0.7925,0.3604,0.3670,0.3620,0.6206
7,Wrapper,Decision Tree,15.0,0.7782,0.3358,0.3879,0.3594,0.6205
8,Univariate,Neural Network,15.0,0.8163,0.4167,0.3293,0.3642,0.6197
9,All Features,SVM,51.0,0.8653,0.8363,0.2241,0.3461,0.6064


## 4. Best Classifier per Feature Strategy

For each feature-selection strategy, the classifier with the highest F1-score
is identified.

This provides a concise comparison of the strongest configuration obtained
under each experimental setting.

In [6]:
best_per_experiment = (
    all_results
    .sort_values("F1", ascending=False)
    .groupby("Experiment", as_index=False)
    .first()
)

display(
    best_per_experiment.round(4)
)

,Experiment,Model,Number of Features,Accuracy,Precision,Recall,F1,Balanced Accuracy
0,All Features,Neural Network,51.0,0.8490,0.5492,0.4219,0.4738,0.6765
1,Multivariate,Neural Network,15.0,0.8245,0.4501,0.3534,0.3902,0.6341
2,Univariate,Decision Tree,15.0,0.7789,0.3546,0.4219,0.3809,0.6347
3,Wrapper,RIPPER,2.6,0.7823,0.3640,0.4687,0.3996,0.6557


## 5. Effect of Feature Selection on Each Classifier

The performance of each classifier is compared across the four feature
representations.

This analysis helps determine whether individual classifiers benefit from
feature selection or perform better using the complete representation.

In [7]:
f1_comparison = all_results.pivot(
    index="Model",
    columns="Experiment",
    values="F1"
)

display(
    f1_comparison.round(4)
)

Experiment,All Features,Multivariate,Univariate,Wrapper
Model,,,,
Decision Tree,0.3620,0.3748,0.3809,0.3594
K-NN,0.2115,0.1632,0.2968,0.3146
Neural Network,0.4738,0.3902,0.3642,0.3919
RIPPER,0.2866,0.2280,0.2316,0.3996
SVM,0.3461,0.0910,0.2737,0.1672


In [8]:
balanced_accuracy_comparison = all_results.pivot(
    index="Model",
    columns="Experiment",
    values="Balanced Accuracy"
)

display(
    balanced_accuracy_comparison.round(4)
)

Experiment,All Features,Multivariate,Univariate,Wrapper
Model,,,,
Decision Tree,0.6206,0.6306,0.6347,0.6205
K-NN,0.5577,0.5391,0.5877,0.5959
Neural Network,0.6765,0.6341,0.6197,0.6286
RIPPER,0.5818,0.5653,0.5645,0.6557
SVM,0.6064,0.5239,0.5793,0.5433


## 6. F1-score Comparison

The F1-score of each classifier is visualized across the four experimental
feature representations.

In [9]:
f1_plot = all_results.pivot(
    index="Model",
    columns="Experiment",
    values="F1"
)

ax = f1_plot.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("F1-score by Classifier and Feature Strategy")
plt.xlabel("Classifier")
plt.ylabel("F1-score")
plt.xticks(rotation=0)

plt.legend(
    title="Feature Strategy",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()

plt.savefig(
    "../../figures/pa1/f1_comparison.pdf",
    bbox_inches="tight"
)

plt.show()

C:\Users\adolf\AppData\Local\Temp\ipykernel_45156\2718762569.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Balanced Accuracy Comparison

Balanced accuracy is visualized because it provides a more informative
assessment than standard accuracy when the class distribution is imbalanced.

In [10]:
balanced_plot = all_results.pivot(
    index="Model",
    columns="Experiment",
    values="Balanced Accuracy"
)

ax = balanced_plot.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title(
    "Balanced Accuracy by Classifier and Feature Strategy"
)

plt.xlabel("Classifier")
plt.ylabel("Balanced Accuracy")
plt.xticks(rotation=0)

plt.legend(
    title="Feature Strategy",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()

plt.savefig(
    "../../figures/pa1/balanced_accuracy_comparison.pdf",
    bbox_inches="tight"
)

plt.show()

C:\Users\adolf\AppData\Local\Temp\ipykernel_45156\1560015924.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Dimensionality Reduction

K-NN, Decision Tree, SVM and Neural Network use 51 transformed features
with all predictors and 15 under each selection strategy. RIPPER uses 30
original predictors with all features, 15 transformed components with the
filters, and a mean of 2.6 original predictors with its wrapper. Reductions
are compared only when feature units match.

In [11]:
feature_counts = pd.DataFrame({
    "Experiment": [
        "All Features",
        "Univariate",
        "Multivariate",
        "Wrapper"
    ],
    "Number of Features": [
        51,
        15,
        15,
        15
    ]
})

feature_counts["Reduction (%)"] = (
    1
    - feature_counts["Number of Features"] / 51
) * 100

display(
    feature_counts.round(2)
)

ripper_feature_counts = (
    all_results.loc[
        all_results["Model"] == "RIPPER",
        ["Experiment", "Number of Features"]
    ].copy()
)

ripper_feature_counts["Feature Unit"] = [
    "transformed components" if experiment in ["Univariate", "Multivariate"]
    else "original predictors"
    for experiment in ripper_feature_counts["Experiment"]
]

display(
    ripper_feature_counts.round(2)
)

,Experiment,Number of Features,Reduction (%)
0,All Features,51,0.00
1,Univariate,15,70.59
2,Multivariate,15,70.59
3,Wrapper,15,70.59


,Experiment,Number of Features,Feature Unit
15,Wrapper,2.6,original predictors
3,All Features,30.0,original predictors
9,Univariate,15.0,transformed components
12,Multivariate,15.0,transformed components


## 9. Selected Feature Comparison

The features selected by the three feature-selection strategies are compared.

Because the selectors operate after preprocessing, categorical attributes may
appear as individual one-hot encoded features.

This descriptive overlap concerns full-data fits of the three selectors.
Fold-specific wrapper selections are examined in the model interpretation.

In [12]:
univariate_features = pd.read_csv(
    "../../results/pa1/univariate_feature_scores.csv"
)

univariate_selected = set(
    univariate_features.loc[
        univariate_features["Selected"],
        "Feature"
    ]
)

In [13]:
multivariate_features = pd.read_csv(
    "../../results/pa1/multivariate_feature_relevance.csv"
)

multivariate_selected = set(
    multivariate_features.loc[
        multivariate_features["Selected"],
        "Feature"
    ]
)

In [14]:
wrapper_features = pd.read_csv(
    "../../results/pa1/wrapper_selected_features.csv"
)

wrapper_selected = set(
    wrapper_features.loc[
        wrapper_features["Selected"],
        "Feature"
    ]
)

In [15]:
all_selected_features = sorted(
    univariate_selected
    | multivariate_selected
    | wrapper_selected
)

feature_overlap = pd.DataFrame({
    "Feature": all_selected_features
})

feature_overlap["Univariate"] = (
    feature_overlap["Feature"]
    .isin(univariate_selected)
)

feature_overlap["Multivariate"] = (
    feature_overlap["Feature"]
    .isin(multivariate_selected)
)

feature_overlap["Wrapper"] = (
    feature_overlap["Feature"]
    .isin(wrapper_selected)
)

feature_overlap["Selection Count"] = (
    feature_overlap[
        [
            "Univariate",
            "Multivariate",
            "Wrapper"
        ]
    ]
    .sum(axis=1)
)

feature_overlap = feature_overlap.sort_values(
    [
        "Selection Count",
        "Feature"
    ],
    ascending=[
        False,
        True
    ]
)

display(feature_overlap)

,Feature,Univariate,Multivariate,Wrapper,Selection Count
0,cat__BusinessTravel_Travel_Frequently,True,True,True,3
10,cat__OverTime_Yes,True,True,True,3
13,num__JobInvolvement,True,True,True,3
16,num__MonthlyIncome,True,True,True,3
21,num__StockOptionLevel,True,True,True,3
7,cat__MaritalStatus_Married,False,True,True,2
8,cat__MaritalStatus_Single,True,False,True,2
9,cat__OverTime_No,True,False,True,2
11,num__Age,True,False,True,2
15,num__JobSatisfaction,True,True,False,2


## 10. Consistently Selected Features

Features selected by several independent strategies are particularly
interesting because they indicate employee characteristics that remain useful
under different feature-selection criteria.

In [16]:
selected_by_all = feature_overlap[
    feature_overlap["Selection Count"] == 3
]

selected_by_two_or_more = feature_overlap[
    feature_overlap["Selection Count"] >= 2
]

print("Features selected by all three methods:")
display(selected_by_all)

print("\nFeatures selected by at least two methods:")
display(selected_by_two_or_more)

Features selected by all three methods:


,Feature,Univariate,Multivariate,Wrapper,Selection Count
0,cat__BusinessTravel_Travel_Frequently,True,True,True,3
10,cat__OverTime_Yes,True,True,True,3
13,num__JobInvolvement,True,True,True,3
16,num__MonthlyIncome,True,True,True,3
21,num__StockOptionLevel,True,True,True,3



Features selected by at least two methods:


,Feature,Univariate,Multivariate,Wrapper,Selection Count
0,cat__BusinessTravel_Travel_Frequently,True,True,True,3
10,cat__OverTime_Yes,True,True,True,3
13,num__JobInvolvement,True,True,True,3
16,num__MonthlyIncome,True,True,True,3
21,num__StockOptionLevel,True,True,True,3
7,cat__MaritalStatus_Married,False,True,True,2
8,cat__MaritalStatus_Single,True,False,True,2
9,cat__OverTime_No,True,False,True,2
11,num__Age,True,False,True,2
15,num__JobSatisfaction,True,True,False,2


## 11. Interpretation at Original Variable Level

One-hot encoding creates several transformed features from the same original
categorical variable.

For interpretation, transformed feature names are mapped back to their
original employee characteristics whenever possible.

In [17]:
DATA_PATH = "../../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv"

df = pd.read_csv(DATA_PATH)

columns_to_remove = [
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

df_model = df.drop(columns=columns_to_remove)

original_variables = list(
    df_model.drop(columns=["Attrition"]).columns
)

categorical_features = (
    df_model
    .drop(columns=["Attrition"])
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

def map_to_original_variable(feature_name):
    """
    Map a transformed feature name back to the corresponding
    original predictor.
    """

    clean_name = feature_name.split("__", 1)[-1]

    # Numerical features retain their original names.
    if clean_name in original_variables:
        return clean_name

    # One-hot encoded features start with the original
    # categorical variable name followed by an underscore.
    categorical_matches = [
        col for col in categorical_features
        if clean_name.startswith(col + "_")
    ]

    if categorical_matches:
        return max(
            categorical_matches,
            key=len
        )

    return clean_name

In [18]:
feature_overlap["Original Variable"] = (
    feature_overlap["Feature"]
    .apply(map_to_original_variable)
)

original_variable_overlap = (
    feature_overlap[
        [
            "Original Variable",
            "Univariate",
            "Multivariate",
            "Wrapper"
        ]
    ]
    .groupby("Original Variable")
    .max()
    .reset_index()
)

original_variable_overlap["Selection Count"] = (
    original_variable_overlap[
        [
            "Univariate",
            "Multivariate",
            "Wrapper"
        ]
    ]
    .sum(axis=1)
)

original_variable_overlap = (
    original_variable_overlap
    .sort_values(
        [
            "Selection Count",
            "Original Variable"
        ],
        ascending=[
            False,
            True
        ]
    )
)

display(original_variable_overlap)

,Original Variable,Univariate,Multivariate,Wrapper,Selection Count
1,BusinessTravel,True,True,True,3
5,JobInvolvement,True,True,True,3
7,JobRole,True,True,True,3
9,MaritalStatus,True,True,True,3
10,MonthlyIncome,True,True,True,3
13,OverTime,True,True,True,3
16,StockOptionLevel,True,True,True,3
0,Age,True,False,True,2
8,JobSatisfaction,True,True,False,2
12,NumCompaniesWorked,False,True,True,2


In [19]:
consistent_original_variables = (
    original_variable_overlap[
        original_variable_overlap["Selection Count"] >= 2
    ]
)

display(
    consistent_original_variables
)

,Original Variable,Univariate,Multivariate,Wrapper,Selection Count
1,BusinessTravel,True,True,True,3
5,JobInvolvement,True,True,True,3
7,JobRole,True,True,True,3
9,MaritalStatus,True,True,True,3
10,MonthlyIncome,True,True,True,3
13,OverTime,True,True,True,3
16,StockOptionLevel,True,True,True,3
0,Age,True,False,True,2
8,JobSatisfaction,True,True,False,2
12,NumCompaniesWorked,False,True,True,2


## 12. Feature Selection Frequency

The number of feature-selection strategies selecting each original employee
characteristic is visualized below.

In [20]:
selection_frequency = (
    original_variable_overlap[
        original_variable_overlap["Selection Count"] > 0
    ]
    .sort_values(
        "Selection Count",
        ascending=False
    )
)

plt.figure(figsize=(10, 6))

plt.bar(
    selection_frequency["Original Variable"],
    selection_frequency["Selection Count"]
)

plt.title(
    "Feature Selection Frequency by Original Variable"
)

plt.xlabel("Original Variable")
plt.ylabel("Number of Selection Methods")

plt.xticks(
    rotation=70,
    ha="right"
)

plt.yticks([1, 2, 3])

plt.tight_layout()

plt.savefig(
    "../../figures/pa1/feature_selection_frequency.pdf",
    bbox_inches="tight"
)

plt.show()

C:\Users\adolf\AppData\Local\Temp\ipykernel_45156\1064028647.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13. Final Experimental Comparison

The following table contains all twenty combinations of classifier and feature
strategy used in Practical Application 1.

In [21]:
final_results = all_results.sort_values(
    [
        "F1",
        "Balanced Accuracy"
    ],
    ascending=False
)

display(
    final_results.round(4)
)

,Experiment,Model,Number of Features,Accuracy,Precision,Recall,F1,Balanced Accuracy
0,All Features,Neural Network,51.0,0.8490,0.5492,0.4219,0.4738,0.6765
15,Wrapper,RIPPER,2.6,0.7823,0.3640,0.4687,0.3996,0.6557
16,Wrapper,Neural Network,15.0,0.8401,0.5278,0.3165,0.3919,0.6286
10,Multivariate,Neural Network,15.0,0.8245,0.4501,0.3534,0.3902,0.6341
5,Univariate,Decision Tree,15.0,0.7789,0.3546,0.4219,0.3809,0.6347
11,Multivariate,Decision Tree,15.0,0.7898,0.3652,0.3958,0.3748,0.6306
6,Univariate,Neural Network,15.0,0.8163,0.4167,0.3293,0.3642,0.6197
1,All Features,Decision Tree,51.0,0.7925,0.3604,0.3670,0.3620,0.6206
17,Wrapper,Decision Tree,15.0,0.7782,0.3358,0.3879,0.3594,0.6205
2,All Features,SVM,51.0,0.8653,0.8363,0.2241,0.3461,0.6064


## 14. Export Final Results

The final comparison tables are exported for use in the scientific report.

In [22]:
final_results.to_csv(
    "../../results/pa1/final_results.csv",
    index=False
)

best_per_experiment.to_csv(
    "../../results/pa1/best_per_experiment.csv",
    index=False
)

feature_overlap.to_csv(
    "../../results/pa1/feature_selection_overlap.csv",
    index=False
)

original_variable_overlap.to_csv(
    "../../results/pa1/original_variable_overlap.csv",
    index=False
)

print("Final analysis results saved successfully.")

Final analysis results saved successfully.


## 15. LaTeX Tables

The main experimental tables are exported directly to LaTeX format so they
can be included in the scientific report without manually copying numerical
results.

In [23]:
best_per_experiment.round(4).to_latex(
    "../../report/tables/best_per_experiment.tex",
    index=False,
    escape=True
)

final_results.round(4).to_latex(
    "../../report/tables/final_results.tex",
    index=False,
    escape=True
)

selected_variables_table = consistent_original_variables.rename(
    columns={
        "Original Variable": "Original variable",
        "Univariate": "U",
        "Multivariate": "M",
        "Wrapper": "W",
        "Selection Count": "Count"
    }
).copy()

for column in ["U", "M", "W"]:

    selected_variables_table[column] = (
        selected_variables_table[column]
        .map({True: "Yes", False: "--"})
    )

selected_variables_table.to_latex(
    "../../report/tables/selected_variables.tex",
    column_format="lcccc",
    index=False,
    escape=True
)

print("LaTeX tables exported successfully.")

LaTeX tables exported successfully.


## 16. Main Conclusions

The comparison of the twenty classifier-feature strategy combinations shows
that feature selection does not universally improve predictive performance.
Its effect depends strongly on the classification algorithm.

- The strongest overall configuration is the **Neural Network using all
  transformed features**, which achieves an F1-score of **0.4738** and a
  balanced accuracy of **0.6765**. It therefore provides the best overall
  balance between identifying attrition cases and correctly classifying both
  target classes.

- The **SVM using all features** achieves the highest standard accuracy
  (**0.8653**) and precision (**0.8363**). However, its recall is only
  **0.2241**, showing that high accuracy alone is misleading for this
  imbalanced classification problem.

- For K-NN, Decision Tree, SVM and Neural Network, all three strategies reduce the transformed
  representation from **51 to 15 features**, corresponding to a dimensionality
  reduction of **70.59%**.

- Among the reduced representations, the **Wrapper + RIPPER** configuration
  obtains the highest F1-score (**0.3996**) and balanced accuracy (**0.6557**).
  It retains a mean of **2.6 original predictors** across the outer folds.

- RIPPER obtains F1 **0.2866** with all original predictors, **0.2316** with
  univariate filtering, and **0.2280** with multivariate filtering.

- Feature selection has different effects depending on the classifier.
  In particular, **K-NN benefits considerably from dimensionality reduction**.
  Its F1-score increases from **0.2115** with all features to **0.3146** with
  wrapper selection.

- The Decision Tree also benefits moderately from univariate selection, while
  the SVM performs consistently worse after dimensionality reduction.

- Although the reduced feature subsets do not outperform the strongest
  full-feature model, they provide substantially simpler representations while
  retaining part of the predictive capability. Therefore, feature selection
  introduces a trade-off between predictive performance, dimensionality, and
  interpretability.

In the original descriptive selections, several characteristics are selected
by all three feature-selection strategies:

- `BusinessTravel`
- `JobInvolvement`
- `JobRole`
- `MaritalStatus`
- `MonthlyIncome`
- `OverTime`
- `StockOptionLevel`

In addition, `Age`, `JobSatisfaction`, `NumCompaniesWorked`, and
`TotalWorkingYears` are selected by at least two of the three strategies.

These repeated selections suggest that working conditions, job involvement,
compensation, professional characteristics, and employment history contain
particularly useful information for distinguishing between employees who
remain in the company and those who leave.

Overall, the complete feature representation provides the strongest predictive
performance for this dataset, while feature-selection methods provide more
compact and interpretable representations whose effectiveness depends on the
classifier being used.

RIPPER's original-predictor counts differ from the filters' transformed-feature
counts. Its wrapper uses a bounded three-step search with two internal folds,
so comparisons also reflect different representations and selection budgets.

## 17. RIPPER Rules and Fold-Specific Selections

The fold-level results provide rule sets, numerical bins, selected inputs and
wrapper traces for interpreting RIPPER predictions. These are training-fold models, not full-data explanations.
An AND joins conditions within a rule; OR joins rules. A matching rule predicts
attrition, otherwise the prediction is negative. Conditions describe predictive
associations in this synthetic dataset, without establishing causality.

In [24]:
import json
ripper_fold_results = pd.read_csv("../../results/pa1/ripper_fold_results.csv")
assert len(ripper_fold_results) == 40
display(ripper_fold_results[["Experiment", "Fold", "Selected Features", "Rules"]])
wrapper_rules = ripper_fold_results.query("Experiment == 'Wrapper'")
print("Illustrative wrapper rules: outer training fold 1")
print(wrapper_rules.iloc[0]["Rules"])
wrapper_frequency = pd.Series(
    [feature for subset in wrapper_rules["Selected Features"]
     for feature in json.loads(subset)]
).value_counts().rename_axis("Original predictor").to_frame("Outer folds selected")
display(wrapper_frequency)


,Experiment,Fold,Selected Features,Rules
0,All Features,1,"[""Age"", ""BusinessTravel"", ""DailyRate"", ""Depart...",[[OverTime=Yes^StockOptionLevel=0^Department=S...
1,All Features,2,"[""Age"", ""BusinessTravel"", ""DailyRate"", ""Depart...",[[OverTime=Yes^JobLevel=1^MonthlyIncome=<2325....
2,All Features,3,"[""Age"", ""BusinessTravel"", ""DailyRate"", ""Depart...",[[JobLevel=1^OverTime=Yes^YearsWithCurrManager...
3,All Features,4,"[""Age"", ""BusinessTravel"", ""DailyRate"", ""Depart...",[[OverTime=Yes^JobLevel=1^YearsWithCurrManager...
4,All Features,5,"[""Age"", ""BusinessTravel"", ""DailyRate"", ""Depart...",[[JobLevel=1^OverTime=Yes^MaritalStatus=Single...
5,All Features,6,"[""Age"", ""BusinessTravel"", ""DailyRate"", ""Depart...",[[OverTime=Yes^MaritalStatus=Single^Department...
6,All Features,7,"[""Age"", ""BusinessTravel"", ""DailyRate"", ""Depart...",[[JobLevel=1^OverTime=Yes^StockOptionLevel=0] ...
7,All Features,8,"[""Age"", ""BusinessTravel"", ""DailyRate"", ""Depart...",[[JobLevel=1^OverTime=Yes^StockOptionLevel=0^E...
8,All Features,9,"[""Age"", ""BusinessTravel"", ""DailyRate"", ""Depart...",[[OverTime=Yes^JobLevel=1^YearsInCurrentRole=<...
9,All Features,10,"[""Age"", ""BusinessTravel"", ""DailyRate"", ""Depart...",[[OverTime=Yes^JobLevel=1^MaritalStatus=Single...


Illustrative wrapper rules: outer training fold 1
[[OverTime=Yes^StockOptionLevel=0^JobRole=SalesExecutive] V [OverTime=Yes^JobRole=LaboratoryTechnician^StockOptionLevel=0] V [OverTime=Yes^StockOptionLevel=0^JobRole=SalesRepresentative] V [StockOptionLevel=0^JobRole=SalesRepresentative] V [StockOptionLevel=0^OverTime=Yes^JobRole=ResearchScientist]]


,Outer folds selected
Original predictor,
OverTime,10
JobRole,8
JobLevel,3
StockOptionLevel,2
YearsAtCompany,1
Department,1
WorkLifeBalance,1
